# Week 4 — Git and version control

**Notebook outcomes**

- Explain what a commit is and why version control matters
- Initialize a repo, stage files, and make commits
- Work with branches, merges, and resolve a simple conflict
- Push and pull with a remote (GitHub)
- Open a pull request


## The problem version control solves

You'll know the situation:

```
paper_final.tex
paper_final_v2.tex
paper_final_v2_chase.tex
paper_final_v2_chase_ACTUAL_FINAL.tex
```

Multiply by a codebase, a co-author, and six months. This is miserable.

**Git** replaces this with:

- Every saved state is tracked.
- Every saved state has a message explaining *why*.
- Any saved state can be restored.
- Two people can work in parallel and merge their changes.

The price is learning some vocabulary and a handful of commands.


## Vocabulary

- **Repository ("repo")** — a directory tracked by Git, with a `.git`
  subdirectory holding the history.
- **Working tree** — your files on disk right now.
- **Staging area (index)** — a draft of the next commit.
- **Commit** — a snapshot of the staging area, plus a message, author,
  and pointer to its parent.
- **Branch** — a moveable pointer to a commit. `main` is the default.
- **Remote** — another copy of the repo elsewhere (usually GitHub).
- **Clone** — copy a remote repo locally.
- **Push / pull** — send / fetch commits to / from the remote.


## Configure git once

Before your first commit, tell Git who you are:

```sh
git config --global user.name  "Your Name"
git config --global user.email "you@example.com"
```

Set your default branch name to `main`:

```sh
git config --global init.defaultBranch main
```

Pick an editor for commit messages:

```sh
git config --global core.editor "code --wait"
```

Check your settings:

```sh
git config --list --show-origin
```


## Your first repository

Let's walk through it. From a terminal:

```sh
mkdir ~/git-sandbox
cd ~/git-sandbox
git init
```

`git init` creates a hidden `.git/` directory. That's it — you have a
repository.

Create a file:

```sh
echo "# My sandbox" > README.md
```

Check the status:

```sh
git status
```

Git tells you `README.md` is **untracked**.


### Stage and commit

```sh
git add README.md         # move README.md into the staging area
git status                # now "Changes to be committed"
git commit -m "initial commit with README"
git log                   # see your commit
```

`git add` stages. `git commit` snapshots the staged state and attaches
your message. `git log` shows the history.


## Seeing what changed

```sh
git status              # what's staged/unstaged/untracked
git diff                # unstaged changes vs. working tree
git diff --staged       # staged changes vs. last commit
git log                 # history
git log --oneline       # one line per commit
git log -p              # with patches
git show <hash>         # view a specific commit
```

`git status` is your most-used command after `cd`. Run it constantly.


## Undoing things

Git has a deserved reputation for scary undo semantics. A few safe ones
to start with:

```sh
# Discard unstaged changes to a tracked file (DESTRUCTIVE)
git restore path/to/file

# Unstage a file (keep the edits on disk)
git restore --staged path/to/file

# Amend the most recent commit (e.g., fix the message or add a file)
# DO NOT amend commits you've already pushed to a shared branch
git commit --amend
```

The deeper tools (`reset`, `revert`, `reflog`, `rebase -i`) we'll leave
for later. For now, follow one rule: **commit early, commit often**. Many
small commits are easy to untangle. One big commit is not.


## Branches

A branch is just a moveable name for a commit. Start from `main` for a
new feature:

```sh
git switch -c feature/add-intro    # create and switch
# ... edit, add, commit ...
git switch main                     # back to main
git merge feature/add-intro         # bring the work in
git branch -d feature/add-intro     # delete the branch (safely)
```

**Why bother?** Two reasons:

1. Pull-request workflow expects a branch per change.
2. You can park unfinished work on a branch and keep `main` clean.


### Merge conflicts

Sometimes Git can't combine changes automatically. You'll see:

```
CONFLICT (content): Merge conflict in paper.tex
```

Open the conflicting file. Git has inserted markers:

```
<<<<<<< HEAD
The world is flat.
=======
The world is round.
>>>>>>> feature/round-earth
```

You (the human) edit the file to the correct final state, **remove the
markers**, then:

```sh
git add paper.tex
git commit
```

Merge conflicts are tedious but not scary. The markers show exactly what
each side wanted; your job is to decide.


## Remotes — working with GitHub

So far everything is local. To collaborate, use a **remote**.

1. Create an empty repo on [github.com](https://github.com).
2. Hook up your local repo:

   ```sh
   git remote add origin git@github.com:yourname/git-sandbox.git
   git push -u origin main
   ```

3. On another machine (or for someone else):

   ```sh
   git clone git@github.com:yourname/git-sandbox.git
   ```

From then on:

```sh
git push           # send local commits to origin
git pull           # fetch and merge remote commits
```


## The pull-request workflow

For shared repos, don't push directly to `main`. Instead:

1. `git switch -c fix/typo-in-intro`
2. Make changes, commit.
3. `git push -u origin fix/typo-in-intro`
4. On GitHub: **Open a pull request** from `fix/typo-in-intro` → `main`.
5. Describe *why* in the PR body.
6. Collaborators review. Address comments with more commits on the branch
   (no force-pushing on shared branches).
7. Once approved, merge the PR.

This is how essentially all collaborative research code on GitHub works.


## `.gitignore`

You don't want to commit everything. A `.gitignore` file at the repo root
lists patterns to ignore:

```gitignore
__pycache__/
.venv/
.ipynb_checkpoints/
*.pdf
*.aux
.DS_Store
```

Commit the `.gitignore` itself. Rule of thumb: commit **source** (code,
docs, small data); ignore **generated output** (compiled binaries, build
artifacts, large data).


## Big files and notebooks

Two pitfalls specific to research:

1. **Large data files.** Git doesn't handle them well. Use `git-lfs`
   (Large File Storage) or keep data out of Git entirely (point to S3,
   a shared drive, or a dataset versioning tool).

2. **Notebook diffs.** `.ipynb` files are JSON, so `git diff` on them is
   ugly. Three options:
   - Live with it for short notebooks.
   - Use `nbstripout` to strip outputs before committing (trade off:
     loses the "ran this" record).
   - Use `jupytext` to pair each notebook with a clean `.py` version that
     diffs beautifully.

   For the camp, live with the ugly diffs. We'll revisit in week 11.


## Cheat sheet

```sh
# one-time setup
git config --global user.name "Your Name"
git config --global user.email "you@example.com"

# everyday loop
git status
git add <file>
git commit -m "message"
git push

# branches
git switch -c new-branch
git switch main
git merge new-branch

# collaboration
git clone git@github.com:user/repo.git
git pull
git push -u origin branch-name
```
